# Part 2 — Train, compare, and test the detector

In this notebook, you will train the complete detector and conduct controlled
experiments. You will:

1. train a baseline model and report its validation result;
2. compare three settings of one training hyperparameter;
3. change one aspect of the model architecture;
4. select a model using validation results and evaluate it once on the test set;
5. visualize and analyze several predictions.

Record your results and explanations in [`report.md`](report.md). Do not use
the test set to select a model or tune a threshold.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import os

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import torch

import assignment2 as a2

DATASET_ROOT = "./data/VMRD_COCO"
try:
    import torch_directml
    DEVICE = torch_directml.device()
except ImportError:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 519
BATCH_SIZE = 32
NUM_EPOCHS = 25

train_loader = a2.get_train_loader(DATASET_ROOT, BATCH_SIZE)
val_loader = a2.get_val_loader(DATASET_ROOT, BATCH_SIZE)

print("Device:", DEVICE)
print("Training images:", len(train_loader.dataset))
print("Validation images:", len(val_loader.dataset))

## 1. Baseline model

You have now implemented the Dataset, detector, loss, training loop, and
evaluation loop. The following cell puts these pieces together to train the
baseline model.

Use the following baseline settings:

- model: `ResNet9Detector` with skip connections;
- batch size: `32`;
- number of epochs: `25`;
- optimizer: `AdamW` with `lr=2.5e-3` and `weight_decay=1e-4`;
- scheduler: `OneCycleLR` with `max_lr=2.5e-3`,
  `steps_per_epoch=len(train_loader)`, and `pct_start=0.15`.

Run the provided cell. In your report, record the validation losses,
precision, and recall; include the training-loss curve; and briefly describe
the training behaviour. This baseline is the reference for the controlled
experiments below.


In [ ]:
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

baseline_model = a2.ResNet9Detector().to(DEVICE)
baseline_history = a2.train_detection_model(
    baseline_model,
    train_loader,
    num_epochs=NUM_EPOCHS,
    device=DEVICE,
)
baseline_val_losses, baseline_val_accuracy = a2.evaluate_detection_model(
    baseline_model, val_loader, DEVICE
)

for name, value in baseline_val_losses.items():
    print(f"Validation {name}: {value:.4f}")
print(f"Validation precision: {baseline_val_accuracy['precision']:.3f}")
print(f"Validation recall:    {baseline_val_accuracy['recall']:.3f}")

batches_per_epoch = len(train_loader)
baseline_epoch_loss = np.array(
    baseline_history["total_loss"]
).reshape(NUM_EPOCHS, batches_per_epoch).mean(axis=1)
plt.figure(figsize=(7, 4))
plt.plot(
    range(1, NUM_EPOCHS + 1),
    baseline_epoch_loss,
    marker="o",
)
plt.xlabel("Epoch")
plt.ylabel("Mean training loss")
plt.title("Baseline training loss")
plt.grid(alpha=0.25)
plt.show()

baseline_model = baseline_model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 2. Change one training hyperparameter

Modify **one** training aspect, such as:
- batch size
- learning rate
- optimizer type or hyperparameters
- scheduler type or hyperparameters

Compare the baseline setting with **two** alternative settings.
Only change one aspect at a time so that the comparison is fair.

Report the validation results and plot all three training-loss curves. Explain how the selected hyperparameter affected convergence and validation performance. If the performance does not change, this is also a valid observation, explain why you think that is the case.

In [ ]:
####################################
# Your implementation starts here. #
pass
# Your implementation ends here. #
##################################

## 3. Change one aspect of the model

Modify the model by changing one architectural aspect. Choose **one**
architectural aspect and make one controlled change. Examples include:

- removing or adding skip connections;
- removing normalization or changing the normalization layer;
- adding or removing a layer;
- changing a layer parameter;
- adding dropout.

Keep the data split and training settings identical to the baseline. Report the
validation result, compare the loss curves, and explain why the change may have
helped or hurt performance.

You may use an optional model setting through `**kwargs`, or directly modify
`assignment2/model.py`. For example, the optional `use_skip` setting can
be used to compare the baseline with a model without skip connections.

In [ ]:
####################################
# Your implementation starts here. #
pass
# Your implementation ends here. #
##################################

## 4. Select a model and evaluate the test set

Select a model using **validation results only**. Consider validation loss,
precision, recall, and the tradeoff between precision and recall. You may
choose one of the models above or train one additional model based on what you
learned. Clearly justify your choice in the report.

After making the decision, evaluate the selected model on the test set exactly
once. Report the final test loss, precision, and recall. Do not go back and
change the model or score threshold based on the test result.


In [ ]:
####################################
# Your implementation starts here. #
pass
# Your implementation ends here. #
##################################


## 5. Visualize and analyze predictions

Visualize several test images with both ground-truth and predicted boxes. The
examples do not need to be failures. In the report, discuss what the detector
does well, what errors or limitations you observe, and at least one reasonable
way to improve it.

Dashed red boxes below are ground truth. Solid colored boxes are predictions
after non-maximum suppression. You may adjust the visualization score threshold
for readability, but do not use test images to tune the model itself.

In [ ]:
from PIL import Image

# `selected_model` comes from your model-selection experiment above. If you
# restarted the kernel, restore or retrain that model before running this cell.
selected_model = selected_model.to(DEVICE)
selected_model.eval()

test_loader = a2.get_test_loader(DATASET_ROOT, BATCH_SIZE)
test_annotations = json.load(open(os.path.join(
    DATASET_ROOT, "annotations", "instances_test.json"
)))
category_names = {
    category["id"]: category["name"]
    for category in test_annotations["categories"]
}

# Randomly choose several image records, as in notebook 00.
num_examples = min(6, len(test_annotations["images"]))
example_indices = np.random.default_rng(SEED).choice(
    len(test_annotations["images"]),
    size=num_examples,
    replace=False,
)
example_tensors = torch.stack([
    test_loader.dataset[int(index)][0]
    for index in example_indices
])

with torch.no_grad():
    logits = selected_model(example_tensors.to(DEVICE))
predictions = a2.decode_predictions(
    logits,
    score_threshold=0.40,
    nms_iou_threshold=0.50,
)
colors = plt.get_cmap("tab10")

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for axis, dataset_index, prediction in zip(
    axes.flat, example_indices, predictions
):
    image_info = test_annotations["images"][int(dataset_index)]
    image_id = image_info["id"]
    image_annotations = [
        annotation for annotation in test_annotations["annotations"]
        if annotation["image_id"] == image_id
    ]

    image_path = os.path.join(
        DATASET_ROOT, "images", "test", image_info["file_name"]
    )
    image = Image.open(image_path).convert("RGB")
    width, height = image.size
    axis.imshow(image)

    # Plot the ground-truth annotations using the visualization from notebook 00.
    for annotation in image_annotations:
        x, y, box_width, box_height = annotation["bbox"]
        axis.add_patch(Rectangle(
            (x, y),
            box_width,
            box_height,
            fill=False,
            edgecolor="red",
            linestyle="--",
            linewidth=2,
        ))
        axis.text(
            x,
            y,
            f"GT: {category_names[annotation['category_id']]}",
            color="red",
            fontsize=8,
        )

    # Add the selected model's predictions to the same image.
    for box, score, label in zip(
        prediction["boxes"],
        prediction["scores"],
        prediction["labels"],
    ):
        x1, y1, x2, y2 = box.tolist()
        color = colors(int(label) % 10)
        axis.add_patch(Rectangle(
            (x1 * width, y1 * height),
            (x2 - x1) * width,
            (y2 - y1) * height,
            fill=False,
            edgecolor=color,
            linewidth=2,
        ))
        axis.text(
            x1 * width,
            max(0, y1 * height - 12),
            f"Pred: {category_names[int(label) + 1]} {float(score):.2f}",
            color="white",
            fontsize=8,
            bbox={"facecolor": color, "alpha": 0.9, "pad": 1},
        )

    axis.set_title(f"image_id={image_id} | {len(image_annotations)} objects")
    axis.axis("off")

for axis in axes.flat[num_examples:]:
    axis.axis("off")

fig.suptitle("Ground truth (dashed red) and predictions (solid)")
plt.tight_layout()
plt.show()